In [ ]:
#import Pkg; Pkg.rm("RigorousInvariantMeasures")

In [ ]:
#Pkg.add(path = "/home/isaia/Coding/RigorousInvariantMeasures.jl/")

In [ ]:
abs(0.9*exp(2*pi*im/4))

In [ ]:
using IntervalArithmetic

In [ ]:
r = 0.9
ϕ = Interval(π)/4

In [ ]:
B(z; μ = r*exp(im*ϕ)) = (z*(μ-z))/(1-μ'*z)

In [ ]:
N = 512

interval_angle = [interval(i,i+1)/N for i in 0:N-1]

r_min = 7/8
r_min_image = [B(r_min*exp(2*pi*im*θ)) for θ in interval_angle]

In [ ]:
abs_r_image_min = r_min-maximum(abs.(r_min_image)).lo

In [ ]:
using Plots

r_min_image_x = [mid(real(x)) for x in r_min_image]
r_min_image_y = [mid(imag(x)) for x in r_min_image]

@info r_min, maximum([abs(x).hi for x in r_min_image])

In [ ]:
r_max = 2

r_max_image = [B(r_max*exp(2*pi*im*θ)) for θ in interval_angle]

r_max_image_x = [mid(real(x)) for x in r_max_image]
r_max_image_y = [mid(imag(x)) for x in r_max_image]

@info r_max, minimum([abs(x).lo for x in r_max_image])

abs_r_image_max = minimum(abs.(r_max_image)).lo-r_max

In [ ]:
circle_x = [cos(θ) for θ in 0:0.01:2π]
circle_y = [sin(θ) for θ in 0:0.01:2π]


r_min_circle_x = [r_min*cos(θ) for θ in 0:0.01:2π]
r_min_circle_y = [r_min*sin(θ) for θ in 0:0.01:2π]

r_max_circle_x = [r_max*cos(θ) for θ in 0:0.01:2π]
r_max_circle_y = [r_max*sin(θ) for θ in 0:0.01:2π]


plot(circle_x, circle_y, label = "radius 1", aspect_ratio = :equal)
plot!(r_min_circle_x, r_min_circle_y, color = :red, label = "$r_min")
plot!(r_min_image_x, r_min_image_y, color = :red, label = "")

plot!(r_max_circle_x, r_max_circle_y, color = :green, label = "$r_max")
plot!(r_max_image_x, r_max_image_y, color = :green, label = "")

In [ ]:
maximum(radius_image)

In [ ]:
r = 1.0000001

radius_image_lo = [abs(B(r*exp(2*pi*im*θ))).lo for θ in 0:0.01:2π]

In [ ]:
minimum(radius_image_lo)

In [ ]:
T(x) = angle(B(exp(2π*im*x)))/(2*pi)+0.5

In [ ]:
import Pkg; Pkg.activate(".")
using Plots

In [ ]:
plot(x->mid(T(x)), 0, 1)

In [ ]:
S(x) = mod(1.0+atan((sin(2*pi*x)-r*sin(ϕ))/(cos(2*pi*x)-r*cos(ϕ)))/pi, 1)

In [ ]:
plot!(S, 0, 1)

In [ ]:
using RigorousInvariantMeasures

In [ ]:
D(x) = 0.5+atan((RigorousInvariantMeasures.sinpi(2*x)-r*sin(ϕ))/(RigorousInvariantMeasures.cospi(2*x)-r*cos(ϕ)))/pi

In [ ]:
testD = mod1_dynamic(x->D(x))

In [ ]:
plot(x-> mid(D(Interval(x))), 0, 1)

In [ ]:
using DualNumbers

In [ ]:
Btheta(θ) = B(exp(2*pi*im*θ))
Bthetaprime(θ) = Btheta(Dual(θ, 1)).epsilon

In [ ]:
Btheta(0.1)*Bthetaprime(0.1)'+Bthetaprime(0.1)*Btheta(0.1)'

In [ ]:
N(x) = real(x-(Btheta(x)-1.0)/Bthetaprime(x))

In [ ]:
test(θ) = exp(2*pi*im*θ)
testprime(θ) = test(Dual(θ, 1)).epsilon

In [ ]:
(test(0.0)*testprime(0.0)')'+test(0.0)*testprime(0.0)'

In [ ]:
x = rand(100)

for i in 1:10
    x = N.(x)
end

v = [z<0 ? z+1 : z for z in x]

a = v[1]
b = x[3]

a, b

In [ ]:
D(0)

In [ ]:
Dyn = RigorousInvariantMeasures.PwMap([D, D, D], [0, a, b, 1], [D(0) 1; 0 1; 0 D(1)])

In [ ]:
Ban = RigorousInvariantMeasures.AnalyticFourierBasis.FourierAnalytic(128, 16384)

In [ ]:
Badj = RigorousInvariantMeasures.AdjointFourierBasis.FourierAdjoint(256, 32768)

In [ ]:
Pan = RigorousInvariantMeasures.AnalyticFourierBasis.Dual(Ban, Dyn; ϵ = 0.000000000000000001, max_iter = 100)

In [ ]:
Padj = RigorousInvariantMeasures.AdjointFourierBasis.assemble_standard(Badj, Dyn; ϵ = 0.000000000000000001, max_iter = 100) 

In [ ]:
Pfloat = mid.(real.(Padj))+mid.(imag.(Padj))

In [ ]:
using Pseudospectra

In [ ]:
spectralportrait(Pfloat)

In [ ]:
N(x::Interval; γ = 0.0) = x-(Btheta(Interval(mid(x)))-exp(2*pi*im*γ))/real(Bthetaprime(x))

In [ ]:
function Newton(t::Interval; γ = 0.0)

    der = Bthetaprime(t)

    @info der

    der_real = real(der)
    der_imag = imag(der)

    func = :real

    if inf(abs(der_real))>inf(abs(der_imag))
        func = :real
    else
        func = :imag
    end

    @info func, der_real, der_imag

    mt = mid(t)


    if func == :real 
        return mt-real(Btheta(Interval(t))-1.0)/der_real ∩ t
    elseif func == :imag
        return mt-imag(Btheta(Interval(t)))/der_imag ∩ t
    end
end

In [ ]:
t = Newton(a)

@info a, t

In [ ]:
t = Newton(t)

In [ ]:
Btheta(t)